# **SNC - UPS System**

### Data Fetching

In [60]:
import psycopg2
import pandas as pd

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

✅ Fetched 7760 rows from 'extraction'


In [61]:
keywords = ["UPS"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Signalling-And-Communication') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

len(df)

169

In [62]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

['notification', 'ups_system', 'work_order']


### UPS System

In [63]:
import re
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_value(value):
    """Check if a value is considered 'NA' based on the pattern."""
    if pd.isna(value) or value is None:
        return True
    if isinstance(value, str):
        return bool(pattern_na.match(value))
    return False

def clean_value(value):
    """Recursively converts 'NA' string values in dicts/lists to np.nan."""
    if isinstance(value, dict):
        return {k: clean_value(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [clean_value(v) for v in value]
    elif isinstance(value, str) and is_na_value(value):
        return np.nan
    else:
        return value

def find_na_keys(d, parent=''):
    """Extracts flattened keys whose values are considered 'NA' (including np.nan)."""
    na_keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                na_keys.extend(find_na_keys(v, full_key))
            elif is_na_value(v): # Checks for string 'NA', None, and np.nan
                na_keys.append(full_key)
    return na_keys

def flattened_json(d):
    flat_data = {}

    def _flatten(data, parent_key=''):
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{parent_key}.{k}" if parent_key else k
                
                if isinstance(v, dict):
                    _flatten(v, new_key)
                else:
                    flat_data[new_key] = v
        elif isinstance(data, list):
            for i, item in enumerate(data):
                _flatten(item, f"{parent_key}[{i}]")

    _flatten(d)
    return flat_data

df_ups = df.copy()

df_ups['ups_system'] = df_ups['json_data'].apply(
    lambda x: x.get('ups_system') if isinstance(x, dict) else None
)

df_ups = df_ups[df_ups['ups_system'].notnull()].copy()

df_ups['workorder_id'] = df_ups['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_ups['ups_system'] = df_ups['ups_system'].apply(clean_value)

df_ups['na_keys'] = df_ups['ups_system'].apply(find_na_keys)
na_counter = Counter(k for keys in df_ups['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flattened_json(r) for r in df_ups['ups_system'].fillna({})]
ups_system = pd.DataFrame(flattened_rows) 
ups_system.index = df_ups.index
ups_system['workorder_id'] = df_ups['workorder_id'].astype('Int64')
ups_system['filename'] = df_ups['filename']

for i, col in enumerate(ups_system.columns, start=1):
    print(f"{i:3d}. {col}")
    
    if col in ['workorder_id', 'filename']:
        continue

    non_null_count = ups_system[col].notna().sum()
    print(f"    Total non-null rows: {non_null_count}")

    valid_workorders = (ups_system.loc[ups_system[col].notna(), 'filename'].unique())
    
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
    else:
        print("   No data found for this column across all work orders.")
    print("-" * 80)


  1. station
    Total non-null rows: 167
   Work Orders with data (167): SC_PM_QTR_UPS_4000608017.pdf, SC_PM_QTR_UPS_4000597779.pdf, SC_PM_QTR_UPS_4000576195.pdf, SC_PM_QTR_UPS_4000576144.pdf, SC_PM_QTR_UPS_4000576104.pdf, SC_PM_QTR_UPS_4000530667.pdf, SC_PM_QTR_UPS_4000520523.pdf, SC_PM_QTR_UPS_4000509125.pdf, SC_PM_QTR_UPS_4000608067.pdf, SC_PM_NA_UPS_NA_1 (6).pdf, SC_PM_QTR_UPS_4000668027.pdf, SC_PM_QTR_UPS_4000467087.pdf, SC_PM_QTR_UPS_4000467064.pdf, SC_PM_NA_UPS_NA_10.pdf, SC_PM_QTR_UPS_4000608037.pdf, SC_PM_QTR_UPS_4000446286.pdf, SC_PM_QTR_UPS_4000446335.pdf, SC_PM_QTR_UPS_4000446338.pdf, SC_PM_QTR_UPS_4000446346.pdf, SC_PM_QTR_UPS_4000446356.pdf, SC_PM_QTR_UPS_4000446359.pdf, SC_PM_QTR_UPS_4000446363.pdf, SC_PM_QTR_UPS_4000467069.pdf, SC_PM_QTR_UPS_4000467073.pdf, SC_PM_QTR_UPS_4000467077.pdf, SC_PM_QTR_UPS_4000467093.pdf, SC_PM_QTR_UPS_4000446126.pdf, SC_PM_QTR_UPS_4000467101.pdf, SC_PM_QTR_UPS_4000467110.pdf, SC_PM_QTR_UPS_4000446312.pdf, SC_PM_QTR_UPS_4000480661.pdf, SC_PM

In [64]:
# 1. Start with the raw extracted json
df_ups = df.copy()
df_ups['ups_system'] = df_ups['json_data'].apply(
    lambda x: x.get('ups_system') if isinstance(x, dict) else None
)
df_ups = df_ups[df_ups['ups_system'].notnull()].copy()

# 2. SKIP clean_value for now
# df_ups['ups_system'] = df_ups['ups_system'].apply(clean_value) <-- Comment this out

# 3. Flatten the RAW data
flattened_rows = [flattened_json(r) for r in df_ups['ups_system'].fillna({})]
ups_system_raw = pd.DataFrame(flattened_rows)
ups_system_raw['filename'] = df_ups['filename'].values

# 4. Search for the specific problematic columns
target_cols = [
    'checklist_items.general_inspection.c.readings.38',
    'checklist_items.general_inspection.c.readings.249',
    'checklist_items.general_inspection.c.readings.248',
    'checklist_items.general_inspection.c.readings.2.5',
    'checklist_items.general_inspection.c.readings.-0.5'
]

for col in target_cols:
    if col in ups_system_raw.columns:
        # Find rows where this column is NOT empty
        culprit_files = ups_system_raw.loc[ups_system_raw[col].notna(), 'filename'].unique()
        print(f"Column: {col}")
        print(f"Found in files: {culprit_files}\n")

### Notification and Work Order

In [65]:
import pandas as pd
import numpy as np
import re

df['notification'] = df['json_data'].apply(
    lambda x: x.get('notification') if isinstance(x, dict) else None
)

len(df['notification'])

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']

def is_na_like(val):
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]

df['na_keys'] = df['notification'].apply(find_na_keys)

df_with_na = df[df['na_keys'].apply(lambda x: len(x) > 0)]

pattern = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def clean_value(val):
    """Clean individual values (string, dict, etc.)."""
    if isinstance(val, str):
        return '' if pattern.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

df['notification'] = df['notification'].apply(clean_value)

df['notification'] = df['notification'].replace(np.nan, '', regex=True)

notification_df = pd.json_normalize(df['notification'])

exclude_cols = [
    'approval.for_mcs_use_only',
    'approval.input_by',
    'approval.closed_date',
    'approval.status',
    'approval.for_mcs_use_only.closed_date',
    'approval.for_mcs_use_only.further_work_required',
    'approval.for_mcs_use_only.finished',
    'approval.for_mcs_use_only.unfinished',
    'approval.for_mcs_use_only.cancelled',
    'approval.for_mcs_use_only.further_work',
    'approval.for_mcs_use_only.required',
]

notification_df = notification_df.drop(
    columns=[c for c in exclude_cols if c in notification_df.columns],
    errors='ignore'
)

In [66]:
df['work_order'] = df['json_data'].apply(
    lambda x: x.get('work_order') if isinstance(x, dict) else None
)

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']

def is_na_like(val):
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]

df['na_keys'] = df['work_order'].apply(find_na_keys)

df_with_na = df[df['na_keys'].apply(lambda x: len(x) > 0)]

pattern = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def clean_value(val):
    """Clean individual values (string, dict, etc.)."""
    if isinstance(val, str):
        return '' if pattern.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

df['work_order'] = df['work_order'].apply(clean_value)

df['work_order'] = df['work_order'].replace(np.nan, '', regex=True)

workorder_df = pd.json_normalize(df['work_order'])

exclude_cols = [
    'approval.for_mcs_use_only',
    'approval.input_by',
    'approval.closed_date',
    'approval.status',
    'approval.for_mcs_use_only.closed_date',
    'approval.for_mcs_use_only.further_work_required',
    'approval.for_mcs_use_only.finished',
    'approval.for_mcs_use_only.unfinished',
    'approval.for_mcs_use_only.cancelled',
    'approval.for_mcs_use_only.further_work',
    'approval.for_mcs_use_only.required',
]

workorder_df = workorder_df.drop(
    columns=[c for c in exclude_cols if c in workorder_df.columns],
    errors='ignore'
)

In [67]:
# import os
# import pandas as pd
# from openpyxl import Workbook

# output_path = '../../output/ups_system.xlsx'

# os.makedirs(os.path.dirname(output_path), exist_ok=True)

# if not os.path.exists(output_path):
#     Workbook().save(output_path)

# with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     ups_system.to_excel(writer, index=False, sheet_name='ups_system')

# print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")


In [ ]:
import os
import pandas as pd

output_path = '../../output/ups_system.xlsx'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

if os.path.exists(output_path):
    mode = 'a'
    if_sheet_exists = 'replace'
else:
    mode = 'w'
    if_sheet_exists = None

with pd.ExcelWriter(output_path, engine='openpyxl', mode=mode, if_sheet_exists=if_sheet_exists) as writer:
    notification_df.to_excel(writer, index=False, sheet_name='notification')
    workorder_df.to_excel(writer, index=False, sheet_name='work_order')
    ups_system.to_excel(writer, index=False, sheet_name='ups_system')

print(f"✅ Exported successfully to '{output_path}' ({'replaced existing sheets' if mode == 'a' else 'created new file'})")


✅ Exported successfully to '../../output/ups_system.xlsx' (replaced existing sheets)
